# Step 1: Standardize and Merge the Data
Instead of trying to merge them blindly, we will load each year, pull out only the columns we need, rename them to match a master schema, and add a Year column so we keep everything organized.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import joblib

In [2]:
# Define our standardized master schema
target_columns = [
    'Country', 'Year', 'Happiness_Score', 'GDP_per_Capita', 
    'Social_Support', 'Life_Expectancy', 'Freedom', 'Generosity', 'Corruption'
]

In [3]:
# --- 2015 ---
df_2015 = pd.read_csv('datasets/2015.csv')
df_2015['Year'] = 2015
df_2015 = df_2015.rename(columns={
    'Country': 'Country',
    'Happiness Score': 'Happiness_Score',
    'Economy (GDP per Capita)': 'GDP_per_Capita',
    'Family': 'Social_Support',
    'Health (Life Expectancy)': 'Life_Expectancy',
    'Freedom': 'Freedom',
    'Generosity': 'Generosity',
    'Trust (Government Corruption)': 'Corruption'
})[target_columns]

# --- 2016 ---
df_2016 = pd.read_csv('datasets/2016.csv')
df_2016['Year'] = 2016
df_2016 = df_2016.rename(columns={
    'Country': 'Country',
    'Happiness Score': 'Happiness_Score',
    'Economy (GDP per Capita)': 'GDP_per_Capita',
    'Family': 'Social_Support',
    'Health (Life Expectancy)': 'Life_Expectancy',
    'Freedom': 'Freedom',
    'Generosity': 'Generosity',
    'Trust (Government Corruption)': 'Corruption'
})[target_columns]

# --- 2017 ---
df_2017 = pd.read_csv('datasets/2017.csv')
df_2017['Year'] = 2017
df_2017 = df_2017.rename(columns={
    'Country': 'Country',
    'Happiness.Score': 'Happiness_Score',
    'Economy..GDP.per.Capita.': 'GDP_per_Capita',
    'Family': 'Social_Support',
    'Health..Life.Expectancy.': 'Life_Expectancy',
    'Freedom': 'Freedom',
    'Generosity': 'Generosity',
    'Trust..Government.Corruption.': 'Corruption'
})[target_columns]

# --- 2018 ---
df_2018 = pd.read_csv('datasets/2018.csv')
df_2018['Year'] = 2018
df_2018 = df_2018.rename(columns={
    'Country or region': 'Country',
    'Score': 'Happiness_Score',
    'GDP per capita': 'GDP_per_Capita',
    'Social support': 'Social_Support',
    'Healthy life expectancy': 'Life_Expectancy',
    'Freedom to make life choices': 'Freedom',
    'Generosity': 'Generosity',
    'Perceptions of corruption': 'Corruption'
})[target_columns]

# --- 2019 ---
df_2019 = pd.read_csv('datasets/2019.csv')
df_2019['Year'] = 2019
df_2019 = df_2019.rename(columns={
    'Country or region': 'Country',
    'Score': 'Happiness_Score',
    'GDP per capita': 'GDP_per_Capita',
    'Social support': 'Social_Support',
    'Healthy life expectancy': 'Life_Expectancy',
    'Freedom to make life choices': 'Freedom',
    'Generosity': 'Generosity',
    'Perceptions of corruption': 'Corruption'
})[target_columns]

# Combine all dataframes vertically
master_df = pd.concat([df_2015, df_2016, df_2017, df_2018, df_2019], ignore_index=True)

# Crucial Step: Handle any missing values (especially in Corruption columns)
master_df = master_df.dropna()

# Save this out for your Streamlit app baseline data!
master_df.to_csv('cleaned_happiness_data.csv', index=False)
print(f"Successfully merged data! Total records: {len(master_df)}")

Successfully merged data! Total records: 781


# Step 2: Train the Linear Regression Model
Now that the data is aligned perfectly, we split it into our features ($X$) and our target ($y$).

In [5]:
# 1. Define Features and Target
features = ['GDP_per_Capita', 'Social_Support', 'Life_Expectancy', 'Freedom', 'Corruption']
target = 'Happiness_Score'

X = master_df[features]
y = master_df[target]

# 2. Split the train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Evaluate model perfomance
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print(f"Training R^2 Score: {train_score:.4f}")
print(f"Testing R^2 Score: {test_score:.4f}")

Training R^2 Score: 0.7642
Testing R^2 Score: 0.7400


### What does the $R^2$ Score mean here? 
Because the Gallup World Poll calculates the final score by adding these exact features to an unmeasured benchmark ("Dystopia Residual"), the Linear Regression model should pull an $R^2$ score easily over 0.75 to 0.85. This means the features account for the vast majority of variations in global happiness!

# Step 3: Save the Model for Your Dashboard

In [6]:
# Save the model object to a file
joblib.dump(model, 'happiness_model.pkl')
print("Model saved as 'happiness_model.pkl' successfully!")

Model saved as 'happiness_model.pkl' successfully!


# Step 4: Verify the Model Coefficients (The Policy Insights)
To see exactly how much weight your model gives to each policy choice, look at its coefficients:

In [7]:
# Look at the weights assigned to each feature
for feature, coef in zip(features, model.coef_):
    print(f"{feature}: {coef:+.4f}")
print(f"Intercept (Baseline Dystopia value): {model.intercept_:.4f}")

GDP_per_Capita: +1.0411
Social_Support: +0.6578
Life_Expectancy: +1.0795
Freedom: +1.6574
Corruption: +1.2053
Intercept (Baseline Dystopia value): 2.2392
